# 03 - Train YOLO11n (single class: lettuce)

Trains an Ultralytics **YOLO11n** detector on the LettuceMOTS boxes produced by `01_dataset_prep`. All knobs live in the **config cell** below - nothing is buried in code.

Prereqs: run `01_dataset_prep` first (it writes `croprow/data/lettuce.yaml` and the converted labels). Needs the torch + ultralytics stack; a CUDA GPU is strongly recommended (CPU training of 100 epochs is very slow).

> Left **unrun** on purpose - whoever trains runs it top to bottom.

In [ ]:
import os, sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "croprow" / "utils.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
from croprow import utils as U
CW = REPO_ROOT / "croprow"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

# ===================== CONFIG (edit here only) =====================
MODEL      = "yolo11n.pt"          # base weights (auto-downloaded by ultralytics)
DATA_YAML  = str(DATA_DIR / "lettuce.yaml")

IMGSZ      = 640
EPOCHS     = 100
BATCH      = 16                    # -1 for auto-batch on GPU
LR0        = 0.01                  # initial learning rate
LRF        = 0.01                  # final LR = LR0 * LRF (cosine)
OPTIMIZER  = "auto"               # auto | SGD | Adam | AdamW
PATIENCE   = 30                    # early-stop patience (epochs)
WORKERS    = 8
DEVICE     = 0                     # GPU index, or "cpu"
SEED       = 42

RUN_NAME   = f"train_yolo11n_{IMGSZ}"
# ===================================================================
print("data yaml:", DATA_YAML)
print("run ->", RUNS_DIR / RUN_NAME)

## Environment check

In [ ]:
# This notebook needs the training/inference stack (torch + ultralytics),
# NOT installed in the light 01/02 env. Install into a Python 3.11 venv with
# numpy<2 -- see croprow/requirements-train.txt and croprow/README.md.
try:
    import torch
    from ultralytics import YOLO
    import ultralytics
    print("torch      :", torch.__version__)
    print("ultralytics:", ultralytics.__version__)
    print("CUDA avail :", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Missing training dependency: {e.name}. Install croprow/requirements-train.txt "
        "into a Python 3.11 (numpy<2) venv before running this notebook."
    ) from e

## Train

In [ ]:
model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    lr0=LR0,
    lrf=LRF,
    optimizer=OPTIMIZER,
    patience=PATIENCE,
    workers=WORKERS,
    device=DEVICE,
    seed=SEED,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)
save_dir = Path(model.trainer.save_dir)
print("run dir:", save_dir)

## Validate best weights + copy them into croprow/models/

In [ ]:
import shutil
best = save_dir / "weights" / "best.pt"
print("best weights:", best, "exists:", best.is_file())

MODELS_DIR.mkdir(parents=True, exist_ok=True)
dst = MODELS_DIR / "best.pt"
if best.is_file():
    shutil.copy2(best, dst)
    print("copied ->", dst)

# Metrics on the val split (mAP50, mAP50-95, precision, recall)
m = YOLO(str(best)).val(data=DATA_YAML, imgsz=IMGSZ, device=DEVICE, split="val")
map50, map5095 = float(m.box.map50), float(m.box.map)
precision, recall = float(m.box.mp), float(m.box.mr)
print(f"mAP50={map50:.4f}  mAP50-95={map5095:.4f}  P={precision:.4f}  R={recall:.4f}")

## Log the run to RESULTS.md

Dataset column is `LettuceMOTS-val` - public metrics are logged as their own row (never merged with own-frame results).

In [ ]:
U.append_results_row(
    RESULTS_MD,
    run=RUN_NAME,
    dataset="LettuceMOTS-val",
    map50=map50, map5095=map5095,
    precision=precision, recall=recall,
    epochs=EPOCHS, imgsz=IMGSZ,
)
print(RESULTS_MD.read_text())